In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
from scipy.stats import norm

First align foreign rate and smiles.

In [58]:
smile = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/usdsgd_vol_smile_1y.csv")
foreign_rate = pd.read_csv("SOFR_1y_compounded.csv")
T = 1

# turn vols in decimals!!

def get_extra_smile_features(smile, r_f, T):

    smile.set_index("CalculationDate", inplace=True)
    smile.drop(columns=["Unnamed: 0"], inplace=True)
    r_f.set_index("date", inplace=True)

    r_f_aligned = r_f.reindex(smile.index, method="ffill")

    new_df = smile.copy()

    kurt_list = []
    skew_atm_list = []

    for t in range(smile.shape[0]):

        smile_t = smile.iloc[t]
        r_f_t = r_f_aligned.iloc[t].values / 100
        deltas = pd.to_numeric(smile_t.index, errors='coerce').values
        deltas[8] = 0.5

        # get forward deltas 
        deltas_f = np.exp(r_f_t * T) * deltas

        # get K/F
        K_Fs = np.zeros_like(deltas_f)
        vols_t = smile_t.values / 100

        for i in range(len(deltas_f)):

            D = deltas_f[i]
            vol = vols_t[i]
            vol_current = max(vol, 1e-6) 

            if D < 0:
                term_1 = np.exp(norm.ppf(D + 1) * vol_current * np.sqrt(T) - 0.5 * vol_current**2 * T)
                K_Fs[i] = 1 / term_1

            elif D >= 0:
                term_1 = np.exp(norm.ppf(D) * vol_current * np.sqrt(T) - 0.5 * vol_current**2 * T)
                K_Fs[i] = 1 / term_1
        
        # fit smile
        X = np.column_stack((np.ones_like(K_Fs), K_Fs, K_Fs**2))
        b0, b1, b2 = np.linalg.lstsq(X, vols_t, rcond=None)[0]

        KURT_t = 2 * b2
        SKEW_ATM = b1 + 2 * b2 * 1 # atm forward

        kurt_list.append(KURT_t)
        skew_atm_list.append(SKEW_ATM)

    new_df['KURT'] = kurt_list
    new_df['SKEW_ATM'] = skew_atm_list

    return smile, r_f_aligned, K_Fs, new_df


In [59]:
smile, r_f, K_Fs, new_df = get_extra_smile_features(smile, foreign_rate, T)

In [60]:
new_df.to_csv("usdsgd_vol_smile_1y_extra.csv")